# 08 - Secondary Metrics Analysis (with Threshold Optimization)

Computes secondary evaluation metrics (F1, F2, sensitivity, specificity, precision) 
for all trained models by running inference on the official test set.

**Key improvement:** Threshold optimization on validation set (per pathology) to reflect 
true model capability, not a poor default threshold.

## Why Threshold Matters

At threshold 0.5, models trained on imbalanced multi-label data (like CheXpert with 6-38% 
positive rates) predict conservatively — leading to high specificity but very low sensitivity. 
This underestimates actual model quality. Standard practice is to optimize threshold on 
validation, then apply to test. This is a well-established methodology (Irvin et al., 2019).

## Metrics Reported (per pathology + mean)

For BOTH threshold 0.5 (raw) and optimized threshold (per pathology, maximizes F2 on val):
- **F1-score:** Harmonic mean of precision and recall
- **F2-score:** Weights recall 2× more than precision (medical screening priority)
- **Sensitivity (Recall):** TP / (TP + FN) — actual disease detection rate
- **Specificity:** TN / (TN + FP) — correct rejection rate
- **Precision:** TP / (TP + FP) — trust in positive predictions

## Behaviour

- Auto-discovers all `_best.pt` files
- **Updates existing JSONs** with `test_secondary_metrics` (containing both threshold sets)
- Resume-safe (skips already-processed)
- Aggregates into CSV for downstream analysis

**Time:** ~1-2 min per model (runs val + test inference). Total ~90-180 min for all ~73 models.

In [1]:
import os
import gc
import json
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from torchvision import transforms, models
from sklearn.metrics import f1_score, fbeta_score, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## 1. Configuration

In [2]:
RAW_DIR = os.path.join('..', 'data', 'raw')
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
TEST_DIR = os.path.join('..', 'data', 'test')

BASELINE_MODEL_DIR = os.path.join('..', 'models')
NOISE_MODELS_DIR = os.path.join('..', 'models', 'noise_experiments')
MIT_MODELS_DIR = os.path.join('..', 'models', 'mitigation_experiments')

BASELINE_RESULTS_DIR = os.path.join('..', 'results')
NOISE_RESULTS_DIR = os.path.join('..', 'results', 'noise_experiments')
MIT_RESULTS_DIR = os.path.join('..', 'results', 'mitigation_experiments')

SECONDARY_DIR = os.path.join('..', 'results', 'secondary_metrics')
os.makedirs(SECONDARY_DIR, exist_ok=True)

TARGET_PATHOLOGIES = [
    'Cardiomegaly', 'Edema', 'Consolidation',
    'Atelectasis', 'Pleural Effusion'
]

BATCH_SIZE = 32
IMAGE_SIZE = 224

# Threshold optimization parameters
DEFAULT_THRESHOLD = 0.5  # Standard threshold for comparison
THRESHOLD_SWEEP = np.linspace(0.05, 0.95, 91)  # 91 candidate thresholds
OPTIMIZE_METRIC = 'f2'  # Optimize threshold to maximize F2 (recall-weighted)

print(f'Threshold sweep: {THRESHOLD_SWEEP[0]:.2f} to {THRESHOLD_SWEEP[-1]:.2f} ({len(THRESHOLD_SWEEP)} candidates)')
print(f'Optimizing for: {OPTIMIZE_METRIC.upper()}')

Threshold sweep: 0.05 to 0.95 (91 candidates)
Optimizing for: F2


## 2. Discover Existing Models & Results

In [3]:
models_to_process = []

# ---- Baseline (single model) ----
baseline_pt = os.path.join(BASELINE_MODEL_DIR, 'baseline_clean_best.pt')
baseline_json = os.path.join(BASELINE_RESULTS_DIR, 'baseline_clean_test_results.json')
if os.path.exists(baseline_pt) and os.path.exists(baseline_json):
    models_to_process.append({
        'model_id': 'baseline_clean',
        'category': 'baseline',
        'method': 'baseline',
        'noise_type': 'none',
        'noise_rate': 0.0,
        'seed': 0,
        'pt_path': baseline_pt,
        'json_path': baseline_json,
    })

# ---- Noise experiments ----
if os.path.exists(NOISE_MODELS_DIR):
    for f in sorted(os.listdir(NOISE_MODELS_DIR)):
        if not f.endswith('_best.pt'):
            continue
        name = f.replace('_best.pt', '')
        parts = name.split('_')
        if len(parts) != 3:
            continue
        try:
            noise_type = parts[0]
            noise_rate = int(parts[1]) / 100
            seed = int(parts[2].replace('seed', ''))
        except ValueError:
            continue
        json_path = os.path.join(NOISE_RESULTS_DIR, f'{name}.json')
        if not os.path.exists(json_path):
            continue
        models_to_process.append({
            'model_id': name,
            'category': 'noise',
            'method': 'no_mitigation',
            'noise_type': noise_type,
            'noise_rate': noise_rate,
            'seed': seed,
            'pt_path': os.path.join(NOISE_MODELS_DIR, f),
            'json_path': json_path,
        })

# ---- Mitigation experiments ----
if os.path.exists(MIT_MODELS_DIR):
    METHOD_PREFIXES = ['label_smoothing', 'mixup', 'coteaching', 'confident_learning']
    for f in sorted(os.listdir(MIT_MODELS_DIR)):
        if not f.endswith('_best.pt'):
            continue
        name = f.replace('_best.pt', '')
        method = None
        rest = None
        for m in METHOD_PREFIXES:
            if name.startswith(m + '_'):
                method = m
                rest = name[len(m) + 1:]
                break
        if method is None:
            continue
        parts = rest.split('_')
        if len(parts) != 3:
            continue
        try:
            noise_type = parts[0]
            noise_rate = int(parts[1]) / 100
            seed = int(parts[2].replace('seed', ''))
        except ValueError:
            continue
        json_path = os.path.join(MIT_RESULTS_DIR, f'{name}.json')
        if not os.path.exists(json_path):
            continue
        models_to_process.append({
            'model_id': name,
            'category': 'mitigation',
            'method': method,
            'noise_type': noise_type,
            'noise_rate': noise_rate,
            'seed': seed,
            'pt_path': os.path.join(MIT_MODELS_DIR, f),
            'json_path': json_path,
        })

print(f'Found {len(models_to_process)} models\n')
categories = {}
for m in models_to_process:
    key = m['method']
    categories[key] = categories.get(key, 0) + 1
print('Breakdown by method:')
for k, v in sorted(categories.items()):
    print(f'  {k:<25s}: {v}')

Found 73 models

Breakdown by method:
  baseline                 : 1
  confident_learning       : 12
  coteaching               : 12
  label_smoothing          : 12
  mixup                    : 12
  no_mitigation            : 24


## 3. Dataset & Model Setup

In [4]:
class CheXpertDataset(Dataset):
    def __init__(self, csv_path, data_dir, pathologies, transform=None):
        self.df = pd.read_csv(csv_path)
        self.data_dir = data_dir
        self.pathologies = pathologies
        self.transform = transform
        self.labels = self.df[pathologies].values.astype(np.float32)
        self.paths = self.df['Path'].values
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img_path = os.path.join(self.data_dir, self.paths[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.from_numpy(self.labels[idx])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation set (for threshold optimization)
val_dataset = CheXpertDataset(
    csv_path=os.path.join(PROCESSED_DIR, 'val_clean.csv'),
    data_dir=RAW_DIR,
    pathologies=TARGET_PATHOLOGIES,
    transform=eval_transform
)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

# Test set (for reporting)
test_dataset = CheXpertDataset(
    csv_path=os.path.join(PROCESSED_DIR, 'test_official.csv'),
    data_dir=TEST_DIR,
    pathologies=TARGET_PATHOLOGIES,
    transform=eval_transform
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Val set: {len(val_dataset)} samples (for threshold optimization)')
print(f'Test set: {len(test_dataset)} samples (for reporting)')

def build_densenet121(num_classes=5):
    model = models.densenet121(weights=None)
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    return model

Val set: 202 samples (for threshold optimization)
Test set: 518 samples (for reporting)


## 4. Core Functions: Get Predictions, Optimize Thresholds, Compute Metrics

In [5]:
@torch.no_grad()
def get_probs_and_labels(model, loader, device):
    """Run inference on loader, return sigmoid probabilities and labels as numpy arrays."""
    model.eval()
    all_logits = []
    all_labels = []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        with autocast('cuda', enabled=device.type == 'cuda'):
            logits = model(images)
        all_logits.append(logits.float().cpu().numpy())
        all_labels.append(labels.numpy())
    all_logits = np.concatenate(all_logits)
    all_labels = np.concatenate(all_labels)
    all_probs = 1 / (1 + np.exp(-all_logits))
    return all_probs, all_labels


def find_optimal_thresholds(val_probs, val_labels, pathologies, threshold_sweep, optimize_metric='f2'):
    """For each pathology, find threshold that maximizes the target metric on validation set.
    
    Returns: dict mapping pathology name -> optimal threshold (float)
    """
    optimal_thresholds = {}
    
    for i, path in enumerate(pathologies):
        yt = val_labels[:, i].astype(int)
        probs = val_probs[:, i]
        
        if len(np.unique(yt)) < 2:
            # Can't optimize if no variation; use default
            optimal_thresholds[path] = 0.5
            continue
        
        best_metric = -1.0
        best_threshold = 0.5
        
        for t in threshold_sweep:
            yp = (probs >= t).astype(int)
            if optimize_metric == 'f2':
                m = fbeta_score(yt, yp, beta=2.0, zero_division=0)
            elif optimize_metric == 'f1':
                m = f1_score(yt, yp, zero_division=0)
            else:
                raise ValueError(f'Unknown metric: {optimize_metric}')
            
            if m > best_metric:
                best_metric = m
                best_threshold = float(t)
        
        optimal_thresholds[path] = best_threshold
    
    return optimal_thresholds


def compute_metrics_at_thresholds(probs, labels, pathologies, thresholds):
    """Compute F1/F2/sens/spec/prec at given thresholds (dict: pathology -> threshold).
    
    Returns: dict with per_pathology and mean metrics.
    """
    per_pathology = {}
    f1_list, f2_list, sens_list, spec_list, prec_list = [], [], [], [], []
    
    for i, path in enumerate(pathologies):
        threshold = thresholds[path] if isinstance(thresholds, dict) else thresholds
        yt = labels[:, i].astype(int)
        yp = (probs[:, i] >= threshold).astype(int)
        
        if len(np.unique(yt)) < 2:
            per_pathology[path] = {k: float('nan') for k in ['f1', 'f2', 'sensitivity', 'specificity', 'precision']}
            per_pathology[path]['threshold'] = float(threshold)
            continue
        
        try:
            tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        except ValueError:
            tn = fp = fn = tp = 0
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        f1 = f1_score(yt, yp, zero_division=0)
        f2 = fbeta_score(yt, yp, beta=2.0, zero_division=0)
        
        per_pathology[path] = {
            'f1': float(f1), 'f2': float(f2),
            'sensitivity': float(sensitivity),
            'specificity': float(specificity),
            'precision': float(precision),
            'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn),
            'threshold': float(threshold)
        }
        f1_list.append(f1); f2_list.append(f2)
        sens_list.append(sensitivity); spec_list.append(specificity); prec_list.append(precision)
    
    return {
        'per_pathology': per_pathology,
        'mean': {
            'f1': float(np.mean(f1_list)) if f1_list else float('nan'),
            'f2': float(np.mean(f2_list)) if f2_list else float('nan'),
            'sensitivity': float(np.mean(sens_list)) if sens_list else float('nan'),
            'specificity': float(np.mean(spec_list)) if spec_list else float('nan'),
            'precision': float(np.mean(prec_list)) if prec_list else float('nan'),
        }
    }


def log(msg):
    ts = datetime.now().strftime('%H:%M:%S')
    print(f'[{ts}] {msg}')

## 5. Main Processing Loop

For each model:
1. Skip if already has both threshold sets in JSON
2. Load model checkpoint
3. Run inference on **validation set** → find optimal per-pathology thresholds (maximize F2)
4. Run inference on **test set** → compute metrics at BOTH threshold=0.5 AND optimal thresholds
5. Update existing JSON with `test_secondary_metrics` (containing both sets)

In [6]:
updated = 0
skipped = 0
errors = 0

log(f'Starting secondary metrics for {len(models_to_process)} models')
log(f'Threshold optimization: sweep {len(THRESHOLD_SWEEP)} values, maximize {OPTIMIZE_METRIC.upper()} on val set\n')

for i, minfo in enumerate(models_to_process):
    model_id = minfo['model_id']
    with open(minfo['json_path']) as f:
        existing = json.load(f)
    
    # Skip if already has our new format (with threshold optimization)
    if 'test_secondary_metrics' in existing and 'optimized_threshold' in existing.get('test_secondary_metrics', {}):
        skipped += 1
        continue
    
    log(f'[{i+1}/{len(models_to_process)}] {model_id}')
    
    try:
        # Load model
        model = build_densenet121(len(TARGET_PATHOLOGIES)).to(device)
        ckpt = torch.load(minfo['pt_path'], weights_only=False, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        del ckpt
        
        # Get validation predictions (for threshold optimization)
        val_probs, val_labels = get_probs_and_labels(model, val_loader, device)
        
        # Find optimal thresholds per pathology (maximize F2 on validation)
        optimal_thresholds = find_optimal_thresholds(
            val_probs, val_labels, TARGET_PATHOLOGIES,
            THRESHOLD_SWEEP, optimize_metric=OPTIMIZE_METRIC
        )
        
        # Get test predictions
        test_probs, test_labels = get_probs_and_labels(model, test_loader, device)
        
        # Compute metrics at DEFAULT threshold (0.5)
        default_metrics = compute_metrics_at_thresholds(
            test_probs, test_labels, TARGET_PATHOLOGIES, DEFAULT_THRESHOLD
        )
        
        # Compute metrics at OPTIMIZED thresholds
        optimized_metrics = compute_metrics_at_thresholds(
            test_probs, test_labels, TARGET_PATHOLOGIES, optimal_thresholds
        )
        
        # Build combined result
        secondary = {
            'default_threshold': {
                'threshold': DEFAULT_THRESHOLD,
                'per_pathology': default_metrics['per_pathology'],
                'mean': default_metrics['mean']
            },
            'optimized_threshold': {
                'thresholds': optimal_thresholds,  # per-pathology dict
                'per_pathology': optimized_metrics['per_pathology'],
                'mean': optimized_metrics['mean'],
                'optimize_metric': OPTIMIZE_METRIC,
                'note': 'Thresholds learned on validation set to maximize the optimize_metric; then applied to test set'
            }
        }
        
        # Update existing JSON
        existing['test_secondary_metrics'] = secondary
        with open(minfo['json_path'], 'w') as f:
            json.dump(existing, f, indent=2)
        
        updated += 1
        d_f2 = default_metrics['mean']['f2']
        o_f2 = optimized_metrics['mean']['f2']
        d_sens = default_metrics['mean']['sensitivity']
        o_sens = optimized_metrics['mean']['sensitivity']
        log(f'    Default (T=0.5):    F2={d_f2:.3f}  Sens={d_sens:.3f}')
        log(f'    Optimized:          F2={o_f2:.3f}  Sens={o_sens:.3f}  (thresholds: {[f"{t:.2f}" for t in optimal_thresholds.values()]})')
        
        del model, val_probs, val_labels, test_probs, test_labels
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as e:
        log(f'    ERROR: {e}')
        errors += 1
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue

print()
log('=== SUMMARY ===')
log(f'Updated: {updated}')
log(f'Skipped (already done): {skipped}')
log(f'Errors: {errors}')

[10:52:37] Starting secondary metrics for 73 models
[10:52:37] Threshold optimization: sweep 91 values, maximize F2 on val set

[10:52:38] [64/73] mixup_asymmetric_20_seed789
[10:53:01]     Default (T=0.5):    F2=0.367  Sens=0.366
[10:53:01]     Optimized:          F2=0.694  Sens=0.926  (thresholds: ['0.05', '0.27', '0.22', '0.20', '0.27'])
[10:53:01] [65/73] mixup_asymmetric_40_seed132
[10:53:22]     Default (T=0.5):    F2=0.269  Sens=0.266
[10:53:22]     Optimized:          F2=0.662  Sens=0.892  (thresholds: ['0.05', '0.24', '0.25', '0.17', '0.13'])
[10:53:22] [66/73] mixup_asymmetric_40_seed456
[10:53:42]     Default (T=0.5):    F2=0.305  Sens=0.312
[10:53:42]     Optimized:          F2=0.671  Sens=0.939  (thresholds: ['0.05', '0.28', '0.25', '0.13', '0.25'])
[10:53:42] [67/73] mixup_asymmetric_40_seed789
[10:54:03]     Default (T=0.5):    F2=0.330  Sens=0.333
[10:54:03]     Optimized:          F2=0.660  Sens=0.945  (thresholds: ['0.05', '0.31', '0.24', '0.17', '0.17'])
[10:54:03] [

## 6. Build Aggregated CSV

Includes BOTH default (T=0.5) and optimized threshold metrics for cross-cutting analysis.

In [7]:
rows = []
for minfo in models_to_process:
    with open(minfo['json_path']) as f:
        data = json.load(f)
    if 'test_secondary_metrics' not in data:
        continue
    sec = data['test_secondary_metrics']
    
    # Handle both old and new format
    if 'default_threshold' in sec:
        default = sec['default_threshold']
        optimized = sec.get('optimized_threshold', {})
    else:
        # Old format (just T=0.5) - treat as default
        default = sec
        optimized = {}
    
    row = {
        'model_id': minfo['model_id'],
        'category': minfo['category'],
        'method': minfo['method'],
        'noise_type': minfo['noise_type'],
        'noise_rate': minfo['noise_rate'],
        'seed': minfo['seed'],
        'test_mean_auc': data.get('test_mean_auc', float('nan')),
        # Default threshold (0.5) metrics
        'default_mean_f1': default['mean'].get('f1', float('nan')),
        'default_mean_f2': default['mean'].get('f2', float('nan')),
        'default_mean_sensitivity': default['mean'].get('sensitivity', float('nan')),
        'default_mean_specificity': default['mean'].get('specificity', float('nan')),
        'default_mean_precision': default['mean'].get('precision', float('nan')),
        # Optimized threshold metrics
        'opt_mean_f1': optimized.get('mean', {}).get('f1', float('nan')),
        'opt_mean_f2': optimized.get('mean', {}).get('f2', float('nan')),
        'opt_mean_sensitivity': optimized.get('mean', {}).get('sensitivity', float('nan')),
        'opt_mean_specificity': optimized.get('mean', {}).get('specificity', float('nan')),
        'opt_mean_precision': optimized.get('mean', {}).get('precision', float('nan')),
    }
    
    # Per-pathology metrics (default threshold)
    for path in TARGET_PATHOLOGIES:
        p_short = path.replace(' ', '_')
        pp_def = default.get('per_pathology', {}).get(path, {})
        pp_opt = optimized.get('per_pathology', {}).get(path, {})
        for metric in ['f1', 'f2', 'sensitivity', 'specificity', 'precision']:
            row[f'default_{p_short}_{metric}'] = pp_def.get(metric, float('nan'))
            row[f'opt_{p_short}_{metric}'] = pp_opt.get(metric, float('nan'))
        # Store learned threshold too
        row[f'opt_{p_short}_threshold'] = pp_opt.get('threshold', float('nan'))
    
    rows.append(row)

all_df = pd.DataFrame(rows)
all_csv = os.path.join(SECONDARY_DIR, 'all_secondary_metrics.csv')
all_df.to_csv(all_csv, index=False)
print(f'Saved: {all_csv}')
print(f'Total rows: {len(all_df)}\n')

if len(all_df) > 0:
    print('Summary by category and method:')
    print(all_df.groupby(['category', 'method']).size().to_frame('count'))

Saved: ..\results\secondary_metrics\all_secondary_metrics.csv
Total rows: 73

Summary by category and method:
                               count
category   method                   
baseline   baseline                1
mitigation confident_learning     12
           coteaching             12
           label_smoothing        12
           mixup                  12
noise      no_mitigation          24


## 7. Aggregated Summary Tables

Two summaries: default threshold vs optimized threshold. Filters out coteaching@5% (exploratory).

In [8]:
df = pd.read_csv(all_csv)
before = len(df)
excluded = (df['method'] == 'coteaching') & (df['noise_rate'] == 0.05)
df_clean = df[~excluded].copy()
if before - len(df_clean) > 0:
    print(f'Excluded {before - len(df_clean)} coteaching@5% (exploratory)\n')

baseline = df_clean[df_clean['category'] == 'baseline']
noise = df_clean[df_clean['category'] == 'noise']
mit = df_clean[df_clean['category'] == 'mitigation']

# ---- Baseline comparison ----
if len(baseline) > 0:
    b = baseline.iloc[0]
    print('=== BASELINE (clean labels) ===')
    print(f'{"":30s}  {"Default (T=0.5)":<18s}  {"Optimized":<18s}')
    print(f'{"F1 (mean)":30s}: {b["default_mean_f1"]:<18.4f}  {b["opt_mean_f1"]:<18.4f}')
    print(f'{"F2 (mean)":30s}: {b["default_mean_f2"]:<18.4f}  {b["opt_mean_f2"]:<18.4f}')
    print(f'{"Sensitivity (mean)":30s}: {b["default_mean_sensitivity"]:<18.4f}  {b["opt_mean_sensitivity"]:<18.4f}')
    print(f'{"Specificity (mean)":30s}: {b["default_mean_specificity"]:<18.4f}  {b["opt_mean_specificity"]:<18.4f}')
    print(f'{"Precision (mean)":30s}: {b["default_mean_precision"]:<18.4f}  {b["opt_mean_precision"]:<18.4f}')
    print(f'\nLearned thresholds (baseline):')
    for path in TARGET_PATHOLOGIES:
        p_short = path.replace(' ', '_')
        thr = b[f'opt_{p_short}_threshold']
        print(f'  {path:22s}: {thr:.3f}')

# ---- Noise summary ----
if len(noise) > 0:
    print('\n\n=== NOISE EXPERIMENTS (F2 - optimized threshold) ===')
    noise_summary = noise.groupby(['noise_type', 'noise_rate']).agg({
        'test_mean_auc': ['mean', 'std'],
        'default_mean_f2': ['mean', 'std'],
        'opt_mean_f2': ['mean', 'std'],
        'default_mean_sensitivity': ['mean', 'std'],
        'opt_mean_sensitivity': ['mean', 'std'],
    }).round(4)
    print(noise_summary)
    noise_summary.to_csv(os.path.join(SECONDARY_DIR, 'noise_secondary_summary.csv'))

# ---- Mitigation summary ----
if len(mit) > 0:
    print('\n\n=== MITIGATION EXPERIMENTS (F2 - optimized threshold) ===')
    mit_summary = mit.groupby(['method', 'noise_type', 'noise_rate']).agg({
        'test_mean_auc': ['mean', 'std'],
        'opt_mean_f2': ['mean', 'std'],
        'opt_mean_sensitivity': ['mean', 'std'],
        'opt_mean_specificity': ['mean', 'std'],
    }).round(4)
    print(mit_summary)
    mit_summary.to_csv(os.path.join(SECONDARY_DIR, 'mitigation_secondary_summary.csv'))

print(f'\nAll summaries saved to: {SECONDARY_DIR}')

=== BASELINE (clean labels) ===
                                Default (T=0.5)     Optimized         
F1 (mean)                     : 0.3698              0.5531            
F2 (mean)                     : 0.3693              0.6961            
Sensitivity (mean)            : 0.3811              0.8606            
Specificity (mean)            : 0.9482              0.6990            
Precision (mean)              : 0.5868              0.4203            

Learned thresholds (baseline):
  Cardiomegaly          : 0.050
  Edema                 : 0.400
  Consolidation         : 0.220
  Atelectasis           : 0.120
  Pleural Effusion      : 0.140


=== NOISE EXPERIMENTS (F2 - optimized threshold) ===
                      test_mean_auc         default_mean_f2          \
                               mean     std            mean     std   
noise_type noise_rate                                                 
asymmetric 0.05              0.8791  0.0106          0.3992  0.0518   
           

## 8. Key Findings Summary

In [9]:
print('=' * 70)
print('SECONDARY METRICS - KEY FINDINGS')
print('=' * 70)

if len(baseline) > 0:
    b = baseline.iloc[0]
    print(f'\n1. THRESHOLD OPTIMIZATION IMPACT (baseline):')
    f2_gain = (b['opt_mean_f2'] - b['default_mean_f2']) * 100
    sens_gain = (b['opt_mean_sensitivity'] - b['default_mean_sensitivity']) * 100
    print(f'   F2 improvement:          +{f2_gain:.2f}pp  (from {b["default_mean_f2"]:.3f} to {b["opt_mean_f2"]:.3f})')
    print(f'   Sensitivity improvement: +{sens_gain:.2f}pp  (from {b["default_mean_sensitivity"]:.3f} to {b["opt_mean_sensitivity"]:.3f})')
    print(f'   -> Threshold 0.5 severely underestimates model quality on this task')

if len(noise) > 0 and len(baseline) > 0:
    sym40 = noise[(noise['noise_type'] == 'symmetric') & (noise['noise_rate'] == 0.40)]
    if len(sym40) > 0:
        opt_f2_drop = (baseline.iloc[0]['opt_mean_f2'] - sym40['opt_mean_f2'].mean()) * 100
        opt_sens_drop = (baseline.iloc[0]['opt_mean_sensitivity'] - sym40['opt_mean_sensitivity'].mean()) * 100
        print(f'\n2. EFFECT OF 40% SYMMETRIC NOISE (at optimized threshold):')
        print(f'   F2 drop:          {opt_f2_drop:+.2f}pp')
        print(f'   Sensitivity drop: {opt_sens_drop:+.2f}pp')

if len(mit) > 0:
    sym40_mit = mit[(mit['noise_type'] == 'symmetric') & (mit['noise_rate'] == 0.40)]
    if len(sym40_mit) > 0:
        best_by_f2 = sym40_mit.groupby('method')['opt_mean_f2'].mean().sort_values(ascending=False)
        best_by_sens = sym40_mit.groupby('method')['opt_mean_sensitivity'].mean()
        print(f'\n3. MITIGATION METHODS AT SYM 40% (optimized threshold, ranked by F2):')
        for method, f2 in best_by_f2.items():
            sens = best_by_sens[method]
            print(f'   {method:<22s}: F2={f2:.3f}, Sensitivity={sens:.3f}')

print('\n' + '=' * 70)
print(f'Results saved to: {SECONDARY_DIR}')
print(f'JSONs updated with both default (T=0.5) and optimized threshold metrics')
print('=' * 70)

SECONDARY METRICS - KEY FINDINGS

1. THRESHOLD OPTIMIZATION IMPACT (baseline):
   F2 improvement:          +32.68pp  (from 0.369 to 0.696)
   Sensitivity improvement: +47.95pp  (from 0.381 to 0.861)
   -> Threshold 0.5 severely underestimates model quality on this task

2. EFFECT OF 40% SYMMETRIC NOISE (at optimized threshold):
   F2 drop:          +5.68pp
   Sensitivity drop: -0.82pp

3. MITIGATION METHODS AT SYM 40% (optimized threshold, ranked by F2):
   confident_learning    : F2=0.628, Sensitivity=0.837
   coteaching            : F2=0.624, Sensitivity=0.917
   mixup                 : F2=0.621, Sensitivity=0.880
   label_smoothing       : F2=0.612, Sensitivity=0.848

Results saved to: ..\results\secondary_metrics
JSONs updated with both default (T=0.5) and optimized threshold metrics
